# Redes interativas de emissões

Complemento ao notebook de redes. Fluxo: **P → agregação dos blocos → grafo dirigido NetworkX → PyVis → HTML**. Mantemos as 20 maiores atividades pela soma das linhas e reunimos as demais em Outras. Nenhuma relação positiva entre grupos é filtrada.

Os cálculos econômicos e a agregação ficam neste notebook; a função externa cuida apenas da apresentação. As imagens estáticas e as análises dos 67 setores permanecem no notebook principal.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import IFrame, display
from redes import dados
from redes.redes import carregar_matriz_emissoes, carregar_setores, matriz_para_grafo
from redes.visualizacoes import figura_rede_interativa

## 1. Entradas verificadas

Origem: CSVs exportados pelo notebook da MIP, conferidos pelo manifesto. Dimensões de P: 67 × 67; unidade: Gg de CO₂. Linhas indicam atividades emissoras, colunas indicam destinos da atribuição. A soma das linhas inclui a diagonal.

A verificação de hash é obrigatória: uma entrada divergente deve ser investigada antes da reprodução integrada.

In [ ]:
P = carregar_matriz_emissoes("matriz_emissoes_producao_2015")
setores = carregar_setores("setores_mip_2015")
assert P.index.equals(setores.index) and P.shape == (67, 67)
display(pd.DataFrame([dados.entrada(i) for i in ["matriz_emissoes_producao_2015", "setores_mip_2015"]]))

## 2. Agregação antes da criação do grafo

Somamos os blocos de P nas duas dimensões, preservando o total. A matriz agregada tem 21 × 21 elementos. As relações entre setores reunidos em Outras passam à diagonal desse grupo. A diagonal é preservada nas contas, mas removida da matriz usada para criar as arestas. Recalculamos a diversidade de destinos entre grupos como o inverso da soma dos quadrados das participações dos destinos.

In [ ]:
redes_visuais = {}
for limite in [20]:
    # Seleção pelo total emitido na origem, incluindo a diagonal; desempate por código.
    maiores_emissores = P.sum(axis=1).sort_index().sort_values(ascending=False, kind="stable").head(limite).index
    agrupamento = pd.Series([codigo if codigo in maiores_emissores else "Outras" for codigo in P.index], index=P.index)
    ordem_visual = maiores_emissores.tolist()
    if "Outras" in agrupamento.values:
        ordem_visual.append("Outras")
    
    # Soma dos blocos: linhas e colunas seguem a mesma classificação de grupos.
    P_visual = P.groupby(agrupamento, sort=False).sum()
    P_visual = P_visual.T.groupby(agrupamento, sort=False).sum().T
    P_visual = P_visual.reindex(index=ordem_visual, columns=ordem_visual)
    np.testing.assert_allclose(P_visual.to_numpy().sum(), P.to_numpy().sum())
    diagonal_visual = pd.Series(np.diag(P_visual), index=P_visual.index)
    valores_visuais = P_visual.to_numpy(copy=True)
    np.fill_diagonal(valores_visuais, 0)
    W_visual = pd.DataFrame(valores_visuais, index=ordem_visual, columns=ordem_visual)
    G_visual = matriz_para_grafo(W_visual)
    
    # Indicadores exclusivos do desenho agregado; não substituem as métricas originais.
    metricas_visuais = pd.DataFrame(index=P_visual.index)
    metricas_visuais["descricao"] = setores.reindex(P_visual.index)
    if "Outras" in metricas_visuais.index:
        metricas_visuais.loc["Outras", "descricao"] = f"Outras ({(agrupamento == 'Outras').sum()} atividades)"
    metricas_visuais["emissoes_totais"] = P_visual.sum(axis=1)
    metricas_visuais["diagonal"] = diagonal_visual
    metricas_visuais["forca_saida"] = W_visual.sum(axis=1)
    metricas_visuais["forca_entrada"] = W_visual.sum(axis=0)
    q_visual = W_visual.div(metricas_visuais["forca_saida"].replace(0, np.nan), axis=0).fillna(0)
    metricas_visuais["destinos_efetivos"] = (1 / q_visual.pow(2).sum(axis=1).replace(0, np.nan)).fillna(0)
    # Relações antes intersetoriais que passam a ser internas ao grupo Outras.
    peso_interno_outras = diagonal_visual.sum() - np.diag(P).sum()
    np.testing.assert_allclose(W_visual.to_numpy().sum() + peso_interno_outras, P.to_numpy().sum() - np.trace(P))
    redes_visuais[limite] = {"P": P_visual, "W": W_visual, "grafo": G_visual,
        "metricas": metricas_visuais, "peso_interno_outras": peso_interno_outras}


## 3. Visualização e exportação

NetworkX fornece o grafo dirigido e as posições iniciais. PyVis desenha nós, conexões e pontas integradas; não calculamos manualmente trajetórias de setas. A área dos nós é proporcional às emissões próprias; a largura das arestas cresce com a raiz quadrada do peso. As duas disposições usam as mesmas escalas.

O posicionamento por forças usa semente 42, sem pesos de atração. A física do navegador fica desativada para preservar o posicionamento inicial. Zoom, arraste e informações ao passar o mouse continuam disponíveis. Distâncias não têm interpretação econômica.

Os HTMLs incluem a biblioteca de redes e são referenciados pelos cards da página: executar esta célula atualiza as visualizações sem reconstruir a página inteira.

In [ ]:
destino = dados.RAIZ_PROJETO / "docs" / "redes_interativas"
destino.mkdir(parents=True, exist_ok=True)
rede = redes_visuais[20]
for organizacao in ["circular", "forcas"]:
    visualizacao = figura_rede_interativa(rede["grafo"], rede["metricas"], organizacao)
    assert len(visualizacao.nodes) == rede["grafo"].number_of_nodes()
    assert len(visualizacao.edges) == rede["grafo"].number_of_edges()
    caminho = destino / f"{organizacao}.html"
    html = visualizacao.generate_html(notebook=False)
    # Títulos são texto simples; CSS preserva suas quebras de linha.
    html = html.replace("</head>", '<style>div.vis-tooltip { white-space: pre-line; max-width: 420px; }</style>\n</head>')
    caminho.write_text(html, encoding="utf-8")
    display(IFrame(src=caminho.relative_to(dados.RAIZ_PROJETO).as_posix(), width="100%", height=800))

## 4. Experimento: movimento e sombras

Esta visualização continua em **2D**. A física do navegador reorganiza os mesmos 21 nós e 420 conexões; sombras dão relevo visual. Arrastar um nó provoca uma nova acomodação da rede. As distâncias e os movimentos não representam grandezas econômicas. Os pesos, cores e tamanhos mantêm as mesmas definições dos cards anteriores.

Use o painel de física abaixo da rede para ativar ou desativar o movimento. O estado final depende da interação e não deve ser interpretado como resultado analítico.

In [ ]:
# Partimos do mesmo grafo; mudam somente as opções do renderizador.
experimento = figura_rede_interativa(rede["grafo"], rede["metricas"], "forcas")
opcoes = json.loads(experimento.options) if isinstance(experimento.options, str) else experimento.options.copy()
opcoes["nodes"]["shadow"] = {"enabled": True, "color": "rgba(0,0,0,0.25)", "size": 12, "x": 5, "y": 5}
opcoes["physics"] = {
    "enabled": True,
    "solver": "barnesHut",
    "barnesHut": {"gravitationalConstant": -12000, "centralGravity": 0.15,
                  "springLength": 180, "springConstant": 0.015, "damping": 0.3,
                  "avoidOverlap": 0.5},
    "stabilization": {"enabled": False},
    "maxVelocity": 15
}
opcoes["configure"] = {"enabled": True, "filter": "physics"}
experimento.options = opcoes
html = experimento.generate_html(notebook=False)
html = html.replace("</head>", '<style>div.vis-tooltip { white-space: pre-line; max-width: 420px; }</style>\n</head>')
caminho = destino / "movimento_sombras.html"
caminho.write_text(html, encoding="utf-8")
display(IFrame(src=caminho.relative_to(dados.RAIZ_PROJETO).as_posix(), width="100%", height=1000))